# Session 2: Python Foundations for Research

**Workshop:** Undergraduate Research Computing Bootcamp

> Open this notebook in Google Colab and run each code cell in order.

## Learning objectives

- Use variables, functions, lists, and dictionaries
- Load and analyze tabular data
- Create simple plots for scientific interpretation
- Check the CPU/GPU resources assigned by Colab
- Use one matrix multiplication example to compare sequential CPU, simple CPU-parallel, and GPU-accelerated workflows
- Explain why parallelization is not always faster

## How to use this notebook

1. Read each short explanation.
2. Run each code cell.
3. Complete the exercises marked **Your turn**.
4. Save a copy of the notebook to your Google Drive or GitHub.

## Python as a research tool

Python is widely used in research because it is readable, flexible, and has a large ecosystem of scientific libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Functions

Functions let us reuse analysis steps and reduce mistakes.

In [ ]:
def compute_mean_and_std(values):
    values = np.asarray(values)
    return values.mean(), values.std()

measurements = [1.2, 1.5, 1.4, 1.8, 1.7]
mean_value, std_value = compute_mean_and_std(measurements)

print("Mean:", mean_value)
print("Standard deviation:", std_value)

## Create a small dataset

The next cell creates a small table that mimics experimental or simulation outputs.

In [ ]:
data = pd.DataFrame({
    "case_id": ["case_01", "case_02", "case_03", "case_04"],
    "pressure_drop": [8.2, 12.5, 6.9, 15.1],
    "flow_rate": [1.1, 1.3, 0.9, 1.5],
})

data

In [ ]:
data["resistance_index"] = data["pressure_drop"] / data["flow_rate"]
data

In [ ]:
plt.figure()
plt.scatter(data["flow_rate"], data["pressure_drop"])
plt.xlabel("Flow rate")
plt.ylabel("Pressure drop")
plt.title("Pressure-flow relationship")
plt.show()

### Your turn

Add a new row to the dataset with your own `case_id`, `pressure_drop`, and `flow_rate`. Recompute the `resistance_index` column and regenerate the plot.

## Computational cost and scaling

Some calculations become much slower as the input size grows. The goal here is not to teach advanced algorithms, but to show students that **problem size matters**.

We will time a simple matrix multiplication. This is a common operation in simulations, image processing, and machine learning.

In [ ]:
import time

def time_matrix_multiply(n, repeats=3, seed=0):
    rng = np.random.default_rng(seed)
    a = rng.random((n, n))
    b = rng.random((n, n))

    durations = []
    for _ in range(repeats):
        start = time.perf_counter()
        c = a @ b
        durations.append(time.perf_counter() - start)

    return min(durations)

sizes = [100, 200, 400, 800, 1200]
times = [time_matrix_multiply(n) for n in sizes]

for n, t in zip(sizes, times):
    print(f"n={n:4d}  time={t:.4f} seconds")

In [ ]:
plt.figure()
plt.plot(sizes, times, marker="o")
plt.xlabel("Matrix size (n x n)")
plt.ylabel("Time (seconds)")
plt.title("Computational cost increases with problem size")
plt.yscale("log")
plt.show()

### Your turn

Change the `sizes` list and rerun the timing cell.

Good starter options:

```python
sizes = [100, 200, 300, 400, 500]
```

Avoid very large values in Colab during a workshop because students may have different runtimes.

## System resources in Colab

In standard Colab, we usually do **not** request an exact number of CPUs or GPUs. Instead, Colab assigns resources to the runtime.

Students should learn to ask:

1. How many CPU cores did this runtime receive?
2. Is a GPU available?
3. Does my code actually use those resources?

In [ ]:
import os
import platform
import sys

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("OS:", platform.platform())
print("CPU cores available:", os.cpu_count())

In [ ]:
# Optional GPU check
# In Colab, go to Runtime > Change runtime type > Hardware accelerator > GPU

try:
    import torch

    if torch.cuda.is_available():
        print("GPU available:", torch.cuda.get_device_name(0))
    else:
        print("No GPU available. To try GPU: Runtime > Change runtime type > GPU")
except Exception as exc:
    print("PyTorch is not available:", exc)

## One example for the rest of the notebook: matrix multiplication

To keep the lesson simple, we will use **one computation** throughout this section: matrix multiplication.

Matrix multiplication is useful because it appears in many research workflows:

- image processing
- machine learning
- simulations
- numerical modeling
- data compression and dimensionality reduction

The goal is not to teach the math in detail. The goal is to see how the **same type of computation** behaves on:

1. a regular CPU workflow,
2. a simple CPU-parallel workflow,
3. a GPU workflow.

In [ ]:
# A small helper function for one independent matrix multiplication task

def one_matrix_multiply_case(n=800, seed=0):
    """Create two random n x n matrices, multiply them, and return one summary value."""
    rng = np.random.default_rng(seed)
    a = rng.random((n, n))
    b = rng.random((n, n))
    c = a @ b

    # Return one number so Python does not need to store all large matrices
    return float(c[0, 0])

# Test one case
one_matrix_multiply_case(n=300, seed=0)

## CPU example: sequential matrix multiplications

Imagine we have several independent cases. For example, these could represent different patients, different simulation settings, or different parameter values.

The sequential version completes one matrix multiplication case at a time.

In [ ]:
# Keep these values moderate for a live Colab workshop.
# Students can increase MATRIX_SIZE later to see how the timing changes.

MATRIX_SIZE = 800
NUM_CASES = 8
seeds = list(range(NUM_CASES))

start = time.perf_counter()

sequential_results = [
    one_matrix_multiply_case(n=MATRIX_SIZE, seed=seed)
    for seed in seeds
]

sequential_time = time.perf_counter() - start

print("Matrix size:", MATRIX_SIZE, "x", MATRIX_SIZE)
print("Number of independent cases:", NUM_CASES)
print("Sequential time:", round(sequential_time, 3), "seconds")
print("First few result values:", sequential_results[:3])

## CPU example: simple parallel version

Now we run the **same matrix multiplication cases**, but we send different cases to different CPU workers.

This is the simplest idea behind parallelization:

> Instead of doing case 1, then case 2, then case 3, we try to work on multiple cases at the same time.

For workshop consistency, we limit the number of workers to avoid overwhelming Colab.

In [ ]:
try:
    from joblib import Parallel, delayed
except ImportError:
    %pip install -q joblib
    from joblib import Parallel, delayed

NUM_WORKERS = min(2, os.cpu_count() or 1)

start = time.perf_counter()

parallel_results = Parallel(n_jobs=NUM_WORKERS)(
    delayed(one_matrix_multiply_case)(n=MATRIX_SIZE, seed=seed)
    for seed in seeds
)

parallel_time = time.perf_counter() - start
speedup = sequential_time / parallel_time

print("Matrix size:", MATRIX_SIZE, "x", MATRIX_SIZE)
print("Number of independent cases:", NUM_CASES)
print("CPU workers used:", NUM_WORKERS)
print("Sequential time:", round(sequential_time, 3), "seconds")
print("Parallel time:", round(parallel_time, 3), "seconds")
print("Speedup:", round(speedup, 2), "x")

if speedup > 1:
    print("The parallel version was faster for this problem size and runtime.")
else:
    print("The parallel version was slower. This often happens when the overhead is larger than the benefit.")

### Important teaching point

Parallelization is not always faster.

The computer has to spend time to:

1. start workers,
2. send work to workers,
3. collect results,
4. combine the outputs.

For small examples, this overhead can be larger than the time saved.

There is another subtle point: NumPy matrix multiplication may already use optimized CPU routines internally. That means each individual matrix multiplication may already be using more than one CPU thread behind the scenes. Running several of them at the same time can sometimes create overhead or competition for the same CPU resources.

The practical lesson is:

> Parallelization helps most when the problem is large enough and naturally separable into independent pieces.

### Your turn: find the break-even point

Try changing these values and rerunning the sequential and parallel cells:

```python
MATRIX_SIZE = 500
MATRIX_SIZE = 800
MATRIX_SIZE = 1200
```

Also try:

```python
NUM_CASES = 4
NUM_CASES = 8
NUM_CASES = 12
```

Do not expect every student to get the same timing. Colab runtimes may have different hardware and background load.

## GPU section: same example, different hardware

Now we keep the same idea — **matrix multiplication** — but move the calculation to a GPU.

In Google Colab:

1. Go to **Runtime**.
2. Select **Change runtime type**.
3. Under **Hardware accelerator**, choose **GPU**.
4. If Colab gives options, choose **T4 GPU**.
5. Click **Save**.

Changing the runtime may restart the notebook. After switching to GPU, rerun the setup cells above or start from this section and run the cells below.

Important: selecting a GPU does not automatically make every Python command faster. The code has to explicitly use the GPU. Here we use PyTorch tensors and send them to `cuda`.

In [ ]:
# Check whether this Colab runtime has a GPU enabled

import torch

print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("No GPU available.")
    print("To enable GPU in Colab: Runtime > Change runtime type > Hardware accelerator > GPU")

## CPU vs GPU matrix multiplication

This cell compares the same matrix multiplication on CPU and GPU.

The first GPU operation may include some setup overhead, so we include a short warm-up before timing the GPU calculation.

In [ ]:
import time
import torch

TORCH_MATRIX_SIZE = 4096

def time_torch_matrix_multiply(n=4096, device="cpu"):
    a = torch.rand((n, n), device=device)
    b = torch.rand((n, n), device=device)

    # Warm-up for GPU timing
    if device == "cuda":
        _ = a @ b
        torch.cuda.synchronize()

    start = time.perf_counter()
    c = a @ b

    if device == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    return elapsed, float(c[0, 0].detach().cpu())

# CPU timing
cpu_time, cpu_value = time_torch_matrix_multiply(n=TORCH_MATRIX_SIZE, device="cpu")

print("Matrix size:", TORCH_MATRIX_SIZE, "x", TORCH_MATRIX_SIZE)
print("CPU time:", round(cpu_time, 3), "seconds")

# GPU timing
if torch.cuda.is_available():
    gpu_time, gpu_value = time_torch_matrix_multiply(n=TORCH_MATRIX_SIZE, device="cuda")
    print("GPU time:", round(gpu_time, 3), "seconds")
    print("GPU speedup:", round(cpu_time / gpu_time, 2), "x")
else:
    print("GPU timing skipped because no GPU is available.")

### What students should notice

The CPU-parallel example may or may not be faster because parallelization has overhead.

The GPU example is usually more impressive for large matrix operations because GPUs are designed to perform many array operations at the same time.

A simple summary:

- **CPU sequential:** one workflow, step by step.
- **CPU parallel:** multiple independent cases at the same time, but with overhead.
- **GPU acceleration:** large matrix/tensor operations performed on specialized hardware.

For research computing, the key question is not simply “Can I parallelize this?”

The better question is:

> Is this problem large enough, independent enough, and structured in a way that the hardware can help?

### Your turn: compare different matrix sizes

Try changing:

```python
TORCH_MATRIX_SIZE = 1024
TORCH_MATRIX_SIZE = 2048
TORCH_MATRIX_SIZE = 4096
```

Then rerun the CPU/GPU timing cell.

Small matrices may not show a large GPU advantage because moving data and setting up GPU work also has overhead. Larger matrix operations usually show the benefit more clearly.

## Wrap-up

In this notebook, we used one example — matrix multiplication — to introduce three ideas:

1. Bigger computational problems take more time.
2. CPU parallelization can help, but it is not always faster.
3. GPUs can greatly accelerate large matrix and tensor operations when the code is written to use them.

Before the next session, make sure you can explain why the parallel CPU version may be slower for small examples, and why the GPU version can be faster for large matrix operations.